In [ ]:
# 计算文本的表征，存成文件
from transformers import AutoTokenizer,AutoModel
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset

model_name = "princeton-nlp/unsup-simcse-bert-base-uncased"
dataset_name = "wiki1m_for_simcse.txt"
dataset = load_dataset("LyuShawn/Dataset-LyuCSE", data_files=dataset_name)
dataset = dataset['train']

output_file = f"data/emb_{model_name.split('/')[-1]}_{dataset_name.split('.')[0]}.npy"

# 采样1000个样本
dataset = dataset.shuffle(seed=42).select(range(10000))
max_seq_length = 32
bs = 1024    # 以bs为单位进行推理
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# 用于存储所有文本的表征
all_embeddings = []

def prepare(examples):

    return tokenizer(examples["text"], padding=False, truncation=True, max_length=max_seq_length)

dataset = dataset.map(prepare, batched=True)


for batch in tqdm(dataset.batch(bs)):
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    # 对齐
    max_len = max([len(ids) for ids in input_ids])
    input_ids = [ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids]
    attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

    input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    last_hidden_state = outputs.last_hidden_state
    pooler_output =last_hidden_state[:,0,:]
    all_embeddings.append(pooler_output.cpu().numpy())

# 将所有的文本表征和成一个array
all_embeddings = np.concatenate(all_embeddings, axis=0)
np.save(output_file, all_embeddings)
print(f"Saved to {output_file}")

In [ ]:
import spacy
from transformers import AutoTokenizer
import numpy as np
from tqdm import tqdm

data_dir = 'draw-data/'

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# 计算原句子
input_file = '../data/wiki1m_for_simcse.txt'
with open(input_file, 'r', encoding='utf-8') as f:
    sent_list = f.read().splitlines()

# 将数据拆分成较小的批次
batch_size = 5000
docs = list(tqdm(nlp.pipe(sent_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=10), total=len(sent_list),desc='计算句子长度'))
sent_l_list = [len(doc) for doc in docs]

sent_l_arr = np.array(sent_l_list)
np.save(data_dir + 'c4-句子长度数组.npy', sent_l_arr)

token_l_list = []

for sent in tqdm(sent_list, desc='计算句子token长度'):
    token_l_list.append(len(tokenizer.tokenize(sent)))

token_l_arr = np.array(token_l_list)
np.save(data_dir + 'c4-句子token长度数组.npy', token_l_arr)

In [ ]:
# 计算title和abstract的长度
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
from knowledge.backend import MySQLClient
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer

data_dir = 'draw-data/'

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

MySQL = MySQLClient()

title_list = []
abstrct_list = []
offset = 0
limit = 1000
print('开始加载数据')
while True:
    sent_list = MySQL.batch_get_page_info_title_abstract(offset,limit)
    offset+=limit
    # if not sent_list:
    #     break
    for sent in sent_list:
        title_list.append(sent[1])
        abstrct_list.append(sent[2])

    break
    # 每10万打印一次
    if offset % 100000 == 0:
        print(f'已加载{offset}条数据')

# 计算title
batch_size = 5000
docs = list(tqdm(nlp.pipe(title_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=10), total=len(title_list),desc='计算title句子长度'))
title_l_list = [len(doc) for doc in docs]

title_sent_l_arr = np.array(title_l_list)
np.save(data_dir + 'c4-title句子长度数组.npy', title_sent_l_arr)

title_token_l_list = []

for sent in tqdm(title_list, desc='计算title句子token长度'):
    title_token_l_list.append(len(tokenizer.tokenize(sent)))

title_token_l_arr = np.array(title_token_l_list)
np.save(data_dir + 'c4-title句子token长度数组.npy', title_token_l_arr)

# 计算abstract
batch_size = 5000
docs = list(tqdm(nlp.pipe(abstrct_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=10), total=len(abstrct_list),desc='计算abstract句子长度'))
abstrct_l_list = [len(doc) for doc in docs]

abstrct_sent_l_arr = np.array(abstrct_l_list)
np.save(data_dir + 'c4-abstrct句子长度数组.npy', abstrct_sent_l_arr)

abstrct_token_l_list = []

for sent in tqdm(abstrct_list, desc='计算title句子token长度'):
    abstrct_token_l_list.append(len(tokenizer.tokenize(sent)))

abstrct_token_l_arr = np.array(abstrct_token_l_list)
np.save(data_dir + 'c4-abstrct句子token长度数组.npy', abstrct_token_l_arr)